In [6]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/baostock')
import datasource
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.get_codes()
code = codes[0]
dt = datasource.get_data(code)
dt.head()

,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,isST
date,,,,,,,,,,,,,
2010-01-04,sh.600000,5.079055,5.088362,4.923170,4.930150,5.046482,66191338,1.419984e+09,2,0.835129,1,-2.3052,0
2010-01-05,sh.600000,4.981336,5.020889,4.837085,4.967376,4.930150,115147943,2.436891e+09,2,1.452808,1,0.7551,0
2010-01-06,sh.600000,4.953417,4.955743,4.858024,4.869658,4.967376,96782575,2.034174e+09,2,1.221095,1,-1.9672,0
2010-01-07,sh.600000,4.858024,4.895251,4.723079,4.760305,4.869658,85236072,1.761801e+09,2,1.075414,1,-2.2456,0
2010-01-08,sh.600000,4.732386,4.839411,4.723079,4.813818,4.760305,65707646,1.349532e+09,2,0.829026,1,1.1241,0


In [7]:
# 我想看看涨停后N天的涨跌幅
MAX_UP = 1.095  # 涨停比例
AFTER = 1       # 涨停后的天数



after = AFTER
dt['isMaxUp'] = dt['close']/dt['preclose'] >= MAX_UP # 是否涨停
dt[f'close{after}'] = dt['close'].shift(-after)      # 将后边n天的收盘价提前
dt2 = dt.iloc[30:,:].loc[dt['isMaxUp']==True, :]     # 绕过前面多少天，然后筛选涨停的
dt2[f'rate{after}'] = dt[f'close{after}'] / dt['close'] # 计算比例
dt2.head()


,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,isST,isMaxUp,close1,rate1
date,,,,,,,,,,,,,,,,
2013-09-09,sh.600000,4.513927,4.623702,4.470018,4.623702,4.202168,580189232,6.069490e+09,2,3.887944,1,10.03130,0,True,5.005717,1.082621
2014-03-21,sh.600000,3.943100,4.342679,3.938709,4.342679,3.947491,577730144,5.543460e+09,2,3.871465,1,10.01110,0,True,4.303160,0.990900
2017-05-25,sh.600000,8.640116,9.507804,8.618056,9.507804,8.640116,222373433,2.803027e+09,2,0.791259,1,10.04256,0,True,9.441624,0.993039


In [8]:
# 这里进行全部股票
lst_dt = []
MAX_UP = 1.095  # 涨停比例
AFTER = 1       # 涨停后的天数
after = AFTER

for i in tqdm(range(len(codes))):
    code = codes[i]
    # 股票
    dt = datasource.get_data(code)
    dt['isMaxUp'] = dt['close']/dt['preclose'] >= MAX_UP # 是否涨停
    dt[f'close{after}'] = dt['close'].shift(-after)      # 将后边n天的收盘价提前
    dt2 = dt.iloc[30:,:].loc[dt['isMaxUp']==True, :]     # 绕过前面多少天，然后筛选涨停的
    dt2[f'rate{after}'] = dt2[f'close{after}'] / dt2['close'] # 计算比例
    lst_dt.append(dt2)

dt_all = pd.concat(lst_dt, ignore_index=True)  # 拼接
print(len(dt_all))
print(dt_all[f'rate{after}'].mean())
print(dt_all[f'rate{after}'].median())


100%|██████████████████████████████████████████████████████████████████████████████| 5493/5493 [01:30<00:00, 60.54it/s]
C:\Users\xuhen\AppData\Local\Temp\ipykernel_28180\1090292326.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dt_all = pd.concat(lst_dt, ignore_index=True)  # 拼接


206639
1.0198313084222828
1.0141375171310052


In [10]:
# 接下来是涨停后5天的涨跌幅把

for i in tqdm(range(len(codes))):
    code = codes[i]
    # 股票
    dt = datasource.get_data(code)
    dt['isMaxUp'] = dt['close']/dt['preclose'] >= MAX_UP # 是否涨停
    for _after in range(5):
        after = _after + 1
        dt[f'close{after}'] = dt['close'].shift(-after)      # 将后边n天的收盘价提前
    dt2 = dt.iloc[30:,:].loc[dt['isMaxUp']==True, :]     # 绕过前面多少天，然后筛选涨停的
    lst_dt.append(dt2)

dt_all = pd.concat(lst_dt, ignore_index=True)  # 拼接
# 计算涨跌幅
for _after in range(5):
    after = _after + 1
    dt_all[f'rate{after}'] = dt_all[f'close{after}'] / dt_all['close']
    print(f'涨停后{after},均值:{dt_all[f'rate{after}'].mean()},中位值:{dt_all[f'rate{after}'].median()},std:{dt_all[f'rate{after}'].std()}')

100%|██████████████████████████████████████████████████████████████████████████████| 5493/5493 [01:31<00:00, 60.04it/s]
C:\Users\xuhen\AppData\Local\Temp\ipykernel_28180\440389338.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dt_all = pd.concat(lst_dt, ignore_index=True)  # 拼接


涨停后1,均值:1.0198313084222828,中位值:1.0141375171310052,std:0.062234527096187105
涨停后2,均值:1.0219103247774908,中位值:1.0067553735926305,std:0.09388872465002297
涨停后3,均值:1.0219062002040136,中位值:1.005012531328321,std:0.11726740232083115
涨停后4,均值:1.0223457786180132,中位值:1.0010529086601738,std:0.1337369345671844
涨停后5,均值:1.0220125774369218,中位值:0.9988545246277204,std:0.14857932326704967
